# Evaluation Summary

This notebook summarizes the evaluation results of the trained models.

## 1. Import Libraries

In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
import pyarrow.parquet as pq
from ipywidgets import interact, FloatSlider
import re


## 2. Define Paths and Load Configurations

In [ ]:
# Define the parent directories for experiments
experiments_base_dir = '/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption/experiments'
output_parent_dirs = [
    os.path.join(experiments_base_dir, d)
    for d in os.listdir(experiments_base_dir)
    if os.path.isdir(os.path.join(experiments_base_dir, d)) and d != 'old'
]


In [ ]:
# Load configs and model paths from all experiments
experiment_data = []

for output_parent_dir in output_parent_dirs:
    experiment_name = os.path.basename(output_parent_dir)
    config_dir = os.path.join(output_parent_dir, 'configs')
    model_dir = os.path.join(output_parent_dir, 'models')
    log_dir = os.path.join(output_parent_dir, 'logs')

    if os.path.exists(model_dir): # Standard case with models directory
        config_files = {}
        if os.path.exists(config_dir):
            config_files = {
                f.split('config_')[-1].replace('.yaml', ''): os.path.join(config_dir, f)
                for f in os.listdir(config_dir) if f.endswith('.yaml')
            }
        
        model_files = [f for f in os.listdir(model_dir) if f.endswith('.keras')]
        for model_filename in model_files:
            model_path = os.path.join(model_dir, model_filename)
            timestamp = model_filename.split('model_')[-1].replace('.keras', '')
            
            config, config_path, train_caseIDs, test_caseIDs = {}, None, [], []
            if timestamp in config_files:
                config_path = config_files[timestamp]
                with open(config_path, 'r') as f:
                    config = yaml.safe_load(f)
                train_caseIDs = config.get('train_caseIDs', [])
                test_caseIDs = config.get('test_caseIDs', [])

            experiment_data.append({
                'experiment_name': experiment_name, 'config_path': config_path,
                'model_path': model_path, 'config': config,
                'train_caseIDs': train_caseIDs, 'test_caseIDs': test_caseIDs
            })
    elif os.path.exists(log_dir) and 'inference' in experiment_name: # Case for inference-only experiments
        log_files = [f for f in os.listdir(log_dir) if f.endswith('.csv')]
        for log_filename in log_files:
            match = re.search(r'(1d_conv|lstm)_(\d{8}_\d{6})', log_filename)
            if match:
                model_type, timestamp = match.groups()
                model_filename = f"{model_type}_model_{timestamp}.keras"
                
                # Reconstruct a plausible model_path, even if it doesn't exist
                # We assume the model would have been in a 'models' dir at the same level as 'logs'
                model_path = os.path.join(os.path.dirname(log_dir), 'models', model_filename)

                experiment_data.append({
                    'experiment_name': experiment_name, 'config_path': None,
                    'model_path': model_path, 'config': {},
                    'train_caseIDs': [], 'test_caseIDs': []
                })


# Create a DataFrame to display the experiment configurations
if experiment_data:
    experiments_df = pd.DataFrame([{
        'Experiment Name': d['experiment_name'],
        'Train Cases': d['train_caseIDs'],
        'Test Cases': d['test_caseIDs'],
        'Model Path': os.path.basename(d['model_path']),
        'Config Path': os.path.basename(d['config_path']) if d['config_path'] else None
    } for d in experiment_data])
else:
    experiments_df = pd.DataFrame()

# Sort by train and test cases for consistent order
if not experiments_df.empty:
    # Convert list to tuple for sorting
    experiments_df['Train Cases'] = experiments_df['Train Cases'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
    experiments_df['Test Cases'] = experiments_df['Test Cases'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
    experiments_df = experiments_df.sort_values(by=['Experiment Name', 'Train Cases', 'Test Cases']).reset_index(drop=True)

experiments_df.head()

In [ ]:
len(experiments_df)

In [ ]:
# Extract timestampID from 'Model Path'
experiments_df['timestampID'] = experiments_df['Model Path'].str.extract(r'(\d{8}_\d{6})')

# Function to extract experiment details from the experiment name
def extract_experiment_details(name):
    parts = name.split('_')
    
    # Default values
    base_data = None
    model_type = None
    learning_mode = None
    transfer_data = None
    
    # Based on the number of parts, extract the information
    if len(parts) >= 3:
        base_data = f"{parts[0]}_{parts[1]}"
        model_type = parts[2]
    
    if 'autoencoder' in name:
        model_type = 'autoencoder'

    if len(parts) > 3:
        if "TL" in parts:
            learning_mode = "transferLearning"
            if len(parts) > 4:
                transfer_data = parts[4]
        elif "inference" in parts:
            learning_mode = "inference"
            if len(parts) > 4:
                transfer_data = parts[4]
    else:
        learning_mode = 'baseTraining'

    return base_data, model_type, learning_mode, transfer_data

# Apply the function to the 'Experiment Name' column
details = experiments_df['Experiment Name'].apply(extract_experiment_details)
experiments_df[['BaseData', 'ModelType', 'LearningMode', 'TransferData']] = pd.DataFrame(details.tolist(), index=experiments_df.index)

# Reorder columns to place new columns after 'Experiment Name'
cols = experiments_df.columns.tolist()
new_cols_order = ['Experiment Name', 'BaseData', 'ModelType', 'LearningMode', 'TransferData']
# Remove the new columns from their current position
for col in new_cols_order[1:]:
    if col in cols:
        cols.remove(col)

# Get the index of 'Experiment Name' and insert the new columns after it
insert_location = cols.index('Experiment Name') + 1
for col in reversed(new_cols_order[1:]):
    cols.insert(insert_location, col)

experiments_df = experiments_df[cols]

experiments_df.head()

In [ ]:
# Prepare to collect evaluation results
eval_results = []
evaluations_to_run = []

# Find the base directory for experiments to locate log files
# This assumes all experiments are in subdirectories of a common 'experiments' folder
base_experiments_dir = os.path.dirname(output_parent_dirs[0])

for index, row in experiments_df.iterrows():
    experiment_name = row['Experiment Name']
    timestamp = row['timestampID']
    
    log_dir = os.path.join(base_experiments_dir, experiment_name, 'logs')
    
    eval_file_found = False
    if os.path.exists(log_dir) and timestamp: # Ensure timestamp is not None
        # Search for any .csv file containing the timestamp
        for f in os.listdir(log_dir):
            if f.endswith('.csv') and timestamp in f:
                eval_file_path = os.path.join(log_dir, f)
                
                # Load the evaluation data
                eval_df = pd.read_csv(eval_file_path)
                
                # Add timestamp and evalFile name for merging
                eval_df['timestampID'] = timestamp
                eval_df['evalFile'] = f
                
                eval_results.append(eval_df)
                eval_file_found = True
                break # Stop after finding the first match
    
    if not eval_file_found:
        # If no file was found, add this experiment to the list of evaluations to run
        evaluations_to_run.append(row.to_dict())

# Merge the evaluation results into the main DataFrame
if eval_results:
    # Concatenate all found evaluation results
    all_evals_df = pd.concat(eval_results, ignore_index=True)
    
    # Merge with the main experiments_df
    experiments_df = pd.merge(experiments_df, all_evals_df, on='timestampID', how='left', suffixes=('', '_eval'))

    # Clean up columns if there are duplicates from the merge
    if 'Train Cases_eval' in experiments_df.columns:
        experiments_df.drop(columns=[col for col in experiments_df.columns if col.endswith('_eval')], inplace=True)

# Display the updated DataFrame
experiments_df.head()

In [ ]:
# Convert the list of missing evaluations to a DataFrame for better readability
if evaluations_to_run:
    evaluations_to_run_df = pd.DataFrame(evaluations_to_run)
    display(evaluations_to_run_df)
else:
    print("No evaluations to run.")

In [ ]:
# Re-run evaluation for missing experiments
# IMPORTANT: Make sure the modelPipelinesTL module is in the python path.
# You might need to add the path to the module to your sys.path
import sys
sys.path.append('/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption/')

# Force reload of the module to pick up changes
import importlib
import modelPipelinesTL
import inspect
importlib.reload(modelPipelinesTL)

from modelPipelinesTL import LSTMPipeline, Conv1DPipeline, AutoencoderPipeline

# --- Diagnostic Step ---
# Inspect the signature of the loaded AutoencoderPipeline.evalModel to confirm the reload worked.
# The signature should show 'history=None', indicating the optional parameter is recognized.
try:
    sig = inspect.signature(AutoencoderPipeline.evalModel)
    print(f"✅ Successfully loaded module. Signature of AutoencoderPipeline.evalModel: {sig}")
    if 'history' not in sig.parameters or sig.parameters['history'].default is inspect.Parameter.empty:
        print("⚠️ Warning: The 'history' parameter in evalModel is not optional. The module may not have reloaded correctly.")
except Exception as e:
    print(f"❌ Error inspecting module: {e}")
# --- End Diagnostic Step ---


if 'evaluations_to_run_df' in locals() and not evaluations_to_run_df.empty:
    print(f"\nRe-running evaluations for {len(evaluations_to_run_df)} experiments...")
    
    for index, row in evaluations_to_run_df.iterrows():
        print(f"\n--- Processing: {row['Experiment Name']} ({row['timestampID']}) ---")
        
        # 1. Create a config dictionary
        config = {}
        if pd.notna(row['Config Path']):
            # This is the original config path from when the model was trained
            full_config_path = os.path.join(experiments_base_dir, row['Experiment Name'], 'configs', row['Config Path'])
            if os.path.exists(full_config_path):
                with open(full_config_path, 'r') as f:
                    config = yaml.safe_load(f)
            else:
                print(f"  [Warning] Config file not found at: {full_config_path}")
        
        # Augment config with info from the dataframe
        config['model_type'] = row['ModelType']
        config['output_parent_dir'] = os.path.join(experiments_base_dir, row['Experiment Name'])
        config['train_caseIDs'] = row['Train Cases']
        config['test_caseIDs'] = row['Test Cases']

        # 2. Instantiate the correct pipeline
        if row['ModelType'] == 'lstm':
            pipeline = LSTMPipeline(config)
        elif row['ModelType'] == '1dconv':
            pipeline = Conv1DPipeline(config)
        elif row['ModelType'] == 'autoencoder':
            pipeline = AutoencoderPipeline(config)
        else:
            print(f"  [Skipping] Unknown model type: {row['ModelType']}")
            continue
            
        # 3. Load the model
        model_path_full = os.path.join(experiments_base_dir, row['Experiment Name'], 'models', row['Model Path'])
        if not os.path.exists(model_path_full):
            print(f"  [Skipping] Model file not found: {model_path_full}")
            continue
        pipeline.model = load_model(model_path_full)
        print(f"  Model loaded from {model_path_full}")

        # 4. Prepare data and Run evaluation
        try:
            if row['ModelType'] in ['lstm', '1dconv']:
                if row['LearningMode'] == 'baseTraining':
                    pipeline.prepare_data(
                        train_case_ids=row['Train Cases'],
                        test_case_ids=row['Test Cases'],
                        test_data_path=config.get('inputPath')
                    )
                elif row['LearningMode'] in ['inference', 'transferLearning']:
                    base_model_experiment = f"{row['BaseData']}_{row['ModelType']}"
                    base_model_row = experiments_df[(experiments_df['Experiment Name'] == base_model_experiment) & (experiments_df['LearningMode'] == 'baseTraining')]
                    if base_model_row.empty:
                        print(f"  [Skipping] Could not find base model for {row['Experiment Name']}")
                        continue
                    base_config_path_name = base_model_row['Config Path'].iloc[0]
                    base_config_path = os.path.join(experiments_base_dir, base_model_experiment, 'configs', base_config_path_name)
                    with open(base_config_path, 'r') as f:
                        base_config = yaml.safe_load(f)
                    scaler_data_path = base_config['inputPath']
                    scaler_cases = base_config['train_caseIDs']
                    test_data_name = row['TransferData']
                    if 'nasa' in row['BaseData']: test_data_path = f"/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/data/00_NASA_Milling/{test_data_name}_milling.parquet"
                    elif 'phm' in row['BaseData']: test_data_path = f"/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/data/01_PHM_Milling/{test_data_name}_milling.parquet"
                    elif 'nature' in row['BaseData']: test_data_path = f"/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/data/02_Nature_Milling/{test_data_name}_milling.parquet"
                    else:
                        print(f"  [Skipping] Unknown base data source for {row['Experiment Name']}")
                        continue
                    if not os.path.exists(test_data_path):
                        print(f"  [Skipping] Could not deduce test data path: {test_data_path}")
                        continue
                    pipeline.prepare_data(train_case_ids=scaler_cases, test_case_ids=row['Test Cases'], scaler_data_path=scaler_data_path, test_data_path=test_data_path)
            
            elif row['ModelType'] == 'autoencoder':
                if row['LearningMode'] == 'baseTraining':
                    # The data path is already in the config from step 1
                    pipeline.prepare_data()
                elif row['LearningMode'] in ['inference', 'transferLearning']:
                    test_data_name = row['TransferData']
                    if 'nasa' in row['BaseData']: test_data_path = f"/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/data/00_NASA_Milling/{test_data_name}_milling.parquet"
                    elif 'phm' in row['BaseData']: test_data_path = f"/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/data/01_PHM_Milling/{test_data_name}_milling.parquet"
                    elif 'nature' in row['BaseData']: test_data_path = f"/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/data/02_Nature_Milling/{test_data_name}_milling.parquet"
                    else:
                        print(f"  [Skipping] Unknown base data source for {row['Experiment Name']}")
                        continue
                    if not os.path.exists(test_data_path):
                        print(f"  [Skipping] Could not deduce test data path: {test_data_path}")
                        continue
                    # Set the inputPath for the pipeline and then call prepare_data
                    pipeline.config['inputPath'] = test_data_path
                    pipeline.prepare_data()

            # 5. Run evaluation
            pipeline.timestamp = row['timestampID']
            if row['ModelType'] == 'autoencoder':
                # Calling evalModel directly, relying on the reload and optional 'history' parameter
                pipeline.evalModel()
            else:
                pipeline.evalModel(mode="evaluation_results")
            print(f"  Evaluation complete for {row['Experiment Name']}.")

        except Exception as e:
            print(f"  [ERROR] Could not run evaluation for {row['Experiment Name']}: {e}")

else:
    print("All experiments have evaluation files. Nothing to re-run.")

In [ ]:
# Save the updated experiments_df with new evaluations
updated_log_path = os.path.join(base_experiments_dir, 'experiments_evaluation_summary.csv')
experiments_df.to_csv(updated_log_path, index=False)

## 3. Visualize Evaluation Results

In [ ]:
# Prepare data for plotting
df = experiments_df.copy()

# Exclude Autoencoders from this analysis
df = df[df['ModelType'] != 'autoencoder'].copy()

# Clean up LearningMode names
df['LearningMode'] = df['LearningMode'].replace({
    'baseTraining': 'Base',
    'inference': 'Inference',
    'transferLearning': 'Transfer Learning'
})

# Clean up ModelType names
df['ModelType'] = df['ModelType'].replace({
    '1dconv': '1D Conv',
    'lstm': 'LSTM'
})

# Convert metrics to numeric, coercing errors
metrics = ['F1-Score', 'Precision', 'Recall', 'Accuracy']
for metric in metrics:
    if metric in df.columns:
        df[metric] = pd.to_numeric(df[metric], errors='coerce')

# Drop rows where metrics are NaN, as they can't be plotted
df.dropna(subset=metrics, inplace=True)

# Define the order for plotting
learning_mode_order = ['Base', 'Inference', 'Transfer Learning']
model_type_order = ['1D Conv', 'LSTM']

df['LearningMode'] = pd.Categorical(df['LearningMode'], categories=learning_mode_order, ordered=True)
df['ModelType'] = pd.Categorical(df['ModelType'], categories=model_type_order, ordered=True)

df.head()

### 3.1. F1-Score Overview Across All Training Modes

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(data=df, x='LearningMode', y='F1-Score', hue='ModelType', order=learning_mode_order, hue_order=model_type_order, palette='viridis')
plt.title('F1-Score by Training Mode and Model Type', fontsize=16)
plt.xlabel('Training Mode', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.legend(title='Model Type')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0, 1)
plt.show()

### 3.2. Detailed Performance by Training Mode

In [ ]:
for mode in learning_mode_order:
    df_mode = df[df['LearningMode'] == mode]
    
    if not df_mode.empty:
        # Melt the DataFrame to have metrics in a single column for easy plotting
        df_melted = df_mode.melt(id_vars=['ModelType', 'BaseData', 'TransferData'], 
                                 value_vars=['F1-Score', 'Precision', 'Recall'],
                                 var_name='Metric', value_name='Score')

        plt.figure(figsize=(15, 8))
        sns.barplot(data=df_melted, x='ModelType', y='Score', hue='Metric', order=model_type_order, palette='plasma')
        plt.title(f'Performance Metrics for {mode} Training', fontsize=16)
        plt.xlabel('Model Type', fontsize=12)
        plt.ylabel('Score', fontsize=12)
        plt.legend(title='Metric')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.ylim(0, 1)
        plt.show()

### 3.3. F1-Score Distribution by Model Type

In [ ]:
plt.figure(figsize=(12, 7))
sns.boxplot(data=df, x='ModelType', y='F1-Score', order=model_type_order, palette='magma')
sns.stripplot(data=df, x='ModelType', y='F1-Score', order=model_type_order, color=".25", alpha=0.6)
plt.title('F1-Score Distribution by Model Type', fontsize=16)
plt.xlabel('Model Type', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0, 1)
plt.show()

## 4. Analysis by Base Dataset

The following plots group the results by the dataset used for training (`BaseData`). For 'Inference' and 'Transfer Learning' modes, the results are further broken down by the dataset they were applied to (`TransferData`).

In [ ]:
# Create a more descriptive column for the experiment scenario
def get_experiment_scenario(row):
    base_data = row['BaseData'].replace('_milling', '')
    transfer_data = row['TransferData']
    
    if row['LearningMode'] == 'Base':
        return f"Train: {base_data}"
    elif pd.notna(transfer_data):
        return f"Train: {base_data} → Test: {transfer_data}"
    else:
        return "N/A"

df['Scenario'] = df.apply(get_experiment_scenario, axis=1)

# Sort scenarios for consistent plotting
df_sorted = df.sort_values(['BaseData', 'TransferData', 'LearningMode'])

# Create the plot
g = sns.catplot(
    data=df_sorted,
    x='Scenario',
    y='F1-Score',
    hue='ModelType',
    col='LearningMode',
    kind='bar',
    col_wrap=1,
    height=6,
    aspect=2.5,
    palette='viridis',
    sharex=False, # Each facet will have its own x-axis labels
    col_order=learning_mode_order
)

# Rotate x-axis labels for better readability
g.set_xticklabels(rotation=45, horizontalalignment='right')

# Set titles and labels
g.fig.suptitle('F1-Score by Dataset Scenario and Training Mode', y=1.03, fontsize=18)
g.set_axis_labels("Experiment Scenario", "F1-Score")
g.set_titles("Training Mode: {col_name}")

# Adjust layout
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

This comprehensive plot allows for a direct comparison of how models trained on a specific dataset perform when tested on others, which is crucial for understanding the effectiveness of transfer learning and inference strategies.

## 5. Accuracy Analysis by Training Mode (Export Ready)

Here are the accuracy results split into separate plots for each training mode, using Arial font for export.

In [ ]:
# Set font properties for export
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

# Ensure the 'Scenario' column exists and data is sorted
if 'Scenario' not in df.columns:
    def get_experiment_scenario(row):
        base_data = row['BaseData'].replace('_milling', '')
        transfer_data = row['TransferData']
        
        if row['LearningMode'] == 'Base':
            return f"Train: {base_data}"
        elif pd.notna(transfer_data):
            return f"Train: {base_data} → Test: {transfer_data}"
        else:
            return "N/A"
    df['Scenario'] = df.apply(get_experiment_scenario, axis=1)

df_sorted = df.sort_values(['BaseData', 'TransferData', 'LearningMode'])

# Generate a separate plot for each learning mode
for mode in learning_mode_order:
    plt.figure(figsize=(14, 8))
    
    # Filter data for the current mode
    data_mode = df_sorted[df_sorted['LearningMode'] == mode].copy()
    
    # Create the bar plot
    ax = sns.barplot(
        data=data_mode,
        x='Scenario',
        y='Accuracy',
        hue='ModelType',
        hue_order=model_type_order,
        palette='viridis'
    )
    
    # Set plot title and labels with appropriate font size
    plt.title(f'Model Accuracy for {mode} Training', fontsize=16)
    plt.xlabel('Experiment Scenario', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    
    # Rotate x-axis labels for readability
    plt.xticks(rotation=45, ha='right')
    
    # Set y-axis limit and add grid
    plt.ylim(0, 1)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Adjust legend
    plt.legend(title='Model Type', fontsize=10)
    
    # Ensure tight layout
    plt.tight_layout()
    
    # Show the plot
    plt.show()

In [ ]:
# Reset font settings to default to avoid affecting other plots in the notebook
plt.rcParams.update(plt.rcParamsDefault)